In [0]:
%pip install -r ../../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from mlflow.models import ModelConfig
from databricks.vector_search.client import VectorSearchClient
import time

In [0]:
dataset_conf_path = "../../conf/dataset.yml"
dataset_conf = ModelConfig(development_config=dataset_conf_path).get("dataset")
vs_conf = dataset_conf.get("vector_indexes")

In [0]:
source_table = dataset_conf.get("tables").get("chapter_chunk_table").get("full_path")
vs_endpoint = vs_conf.get("chapter_index").get("endpoint_name")
vs_index = vs_conf.get("chapter_index").get("full_name")

## Create Vector Index

In [0]:
def index_exists(vsc, endpoint_name, index_full_name):
    try:
        vsc.get_index(endpoint_name, index_full_name).describe()
        return True
    except Exception as e:
        if "RESOURCE_DOES_NOT_EXIST" not in str(e):
            print(
                f"Unexpected error describing the index. This could be a permission issue."
            )
            raise e
    return False
  
def wait_for_index_to_be_ready(vsc, vs_endpoint_name, index_name):
  for i in range(180):
    idx = vsc.get_index(vs_endpoint_name, index_name).describe()
    index_status = idx.get('status', idx.get('index_status', {}))
    status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
    url = index_status.get('index_url', index_status.get('url', 'UNKNOWN'))
    if "ONLINE" in status:
      return
    if "UNKNOWN" in status:
      print(f"Can't get the status - will assume index is ready {idx} - url: {url}")
      return
    elif "PROVISIONING" in status:
      if i % 40 == 0: print(f"Waiting for index to be ready, this can take a few min... {index_status} - pipeline url:{url}")
      time.sleep(10)
    else:
        raise Exception(f'''Error with the index - this shouldn't happen. DLT pipeline might have been killed.\n Please delete it and re-run the previous cell: vsc.delete_index("{index_name}, {vs_endpoint_name}") \nIndex details: {idx}''')
  raise Exception(f"Timeout, your index isn't ready yet: {vsc.get_index(index_name, vs_endpoint_name)}")

In [0]:
vsc = VectorSearchClient()

if not index_exists(vsc, vs_endpoint, vs_index):
  print(f"Creating index {vs_index} on endpoint {vs_endpoint}...")
  vsc.create_delta_sync_index(
    endpoint_name=vs_endpoint,
    index_name=vs_index,
    source_table_name=source_table,
    pipeline_type="TRIGGERED",
    primary_key="chapter",
    embedding_source_column='chapter_text',
    embedding_model_endpoint_name='databricks-gte-large-en'
  )
  #Let's wait for the index to be ready and all our embeddings to be created and indexed
  wait_for_index_to_be_ready(vsc, vs_endpoint, vs_index)
else:
  #Trigger a sync to update our vs content with the new data saved in the table
  wait_for_index_to_be_ready(vsc, vs_endpoint, vs_index)
  vsc.get_index(vs_endpoint, vs_index).sync()

print(f"index {vs_index} on table {source_table} is ready")

## Query the Index

In [0]:
question = "What are the key factors associated with higher banking sector outreach across countries, according to the study?"

results = vsc.get_index(vs_endpoint, vs_index).similarity_search(
  query_text=question,
  columns=["chapter_text", "min_page", "max_page"],
  num_results=1)
docs = results.get('result', {}).get('data_array', [])
docs